In [1]:
# mount drive, set paths, load the raw data
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

BASE = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 4/'
DATA_DIR = BASE + 'data/'
RESULTS_DIR = BASE + 'results/'
SEED = 42

df = pd.read_csv(DATA_DIR + 'networkTraffic.csv')
print("Shape:", df.shape)
print(df['attack_cat'].value_counts().sort_index())

Mounted at /content/drive
Shape: (257673, 44)
attack_cat
0    93000
1    13987
2     2329
3    16353
4    44525
5     2677
6    24246
7      174
8     1511
9    58871
Name: count, dtype: int64


In [2]:
# drop id and fix the two invalid-value issues found in the A2 EDA
# id is unique per row so it carries no information
# nothing else is dropped here - feature removal is model specific and happens later

df = df.drop(columns=['id'])

# state has one row = 'no', which is not a documented category - recode to the mode (FIN)
state_mode = df['state'].mode()[0]
df['state'] = df['state'].replace('no', state_mode)

# is_ftp_login should be binary but 26 rows have 2 or 4
# all 26 have a nonzero ct_ftp_cmd so a login did happen - map to 1
df['is_ftp_login'] = df['is_ftp_login'].replace({2: 1, 4: 1})

print("Shape:", df.shape)
print("state mode used:", state_mode)
print("state fixed:", not (df['state'] == 'no').any())
print("is_ftp_login values:", sorted(df['is_ftp_login'].unique()))

Shape: (257673, 43)
state mode used: FIN
state fixed: True
is_ftp_login values: [np.int64(0), np.int64(1)]


In [3]:
# duplicates and conflicting labels
# checking on the raw values before any encoding or scaling, same reasoning as A2
# two different things here:
#   exact duplicates = same features AND same label
#   conflicts = same features but more than one label, which no model can get fully right

feature_cols = [c for c in df.columns if c != 'attack_cat']

n_exact = df.duplicated().sum()
n_feature_dupes = df.duplicated(subset=feature_cols).sum()
print(f"Exact duplicate rows (features + label): {n_exact} ({n_exact/len(df)*100:.2f}%)")
print(f"Duplicate rows on features only:         {n_feature_dupes} ({n_feature_dupes/len(df)*100:.2f}%)")
print(f"Unique feature vectors:                  {len(df) - n_feature_dupes}")

# number of distinct labels attached to each row's feature vector
df['n_labels'] = df.groupby(feature_cols, sort=False)['attack_cat'].transform('nunique')

conflict_rows = df[df['n_labels'] > 1]
n_conflict_vectors = conflict_rows.drop_duplicates(subset=feature_cols).shape[0]
print(f"\nFeature vectors with more than one label: {n_conflict_vectors}")
print(f"Raw rows involved:                        {len(conflict_rows)}")
print("\nConflict rows per class:")
print(conflict_rows['attack_cat'].value_counts().sort_index())

Exact duplicate rows (features + label): 94928 (36.84%)
Duplicate rows on features only:         103989 (40.36%)
Unique feature vectors:                  153684

Feature vectors with more than one label: 2445
Raw rows involved:                        40400

Conflict rows per class:
attack_cat
0      928
1     2246
2     1977
3    12663
4    16507
5     2061
6     3617
7       15
8       13
9      373
Name: count, dtype: int64


In [4]:
# collapse to one row per unique feature vector
# keeping two candidate labels so the conflict treatment can be compared later:
#   label_first    = label of the first occurrence (what A2 effectively did)
#   label_majority = most common label across all rows sharing that vector
# is_conflict marks the vectors that carried more than one label

df['vec_id'] = df.groupby(feature_cols, sort=False).ngroup()

# count how often each label appears for each vector, ties go to the label seen first
counts = df.groupby(['vec_id', 'attack_cat'], sort=False).size().reset_index(name='n')
counts['order'] = np.arange(len(counts))
majority = (counts.sort_values(['vec_id', 'n', 'order'], ascending=[True, False, True])
                  .drop_duplicates('vec_id')
                  .set_index('vec_id')['attack_cat'])

vec = df.drop_duplicates(subset='vec_id').copy()
vec = vec.rename(columns={'attack_cat': 'label_first'})
vec['label_majority'] = vec['vec_id'].map(majority)
vec['is_conflict'] = (vec['n_labels'] > 1).astype(int)
vec['n_rows'] = vec['vec_id'].map(df['vec_id'].value_counts())
vec = vec.drop(columns=['n_labels', 'vec_id']).reset_index(drop=True)

print("Shape:", vec.shape)
print("Conflicting vectors:", vec['is_conflict'].sum())
print("Vectors where first and majority label differ:", (vec['label_first'] != vec['label_majority']).sum())
print("Raw rows accounted for:", vec['n_rows'].sum())

summary = pd.DataFrame({
    'first': vec['label_first'].value_counts().sort_index(),
    'majority': vec['label_majority'].value_counts().sort_index(),
    'conflicts_removed': vec[vec['is_conflict'] == 0]['label_first'].value_counts().sort_index()
})
print("\nClass sizes under each treatment:")
print(summary)

Shape: (153684, 46)
Conflicting vectors: 2445
Vectors where first and majority label differ: 1384
Raw rows accounted for: 257673

Class sizes under each treatment:
   first  majority  conflicts_removed
0  85646     85594              85308
1   8664      8465               8419
2    425       353                350
3   4304      4286               3624
4  25825     26718              25451
5    810       451                450
6  19072     18908              18749
7    163       163                159
8   1449      1451               1446
9   7326      7295               7283


In [5]:
# stratified 80/20 split, then 5 folds inside the 80%
# the 20% test set is not touched until the final evaluation - in A2 every setting was
# tuned on fold 1 and fold 1 was then also used for evaluation, which is what this fixes
# stratifying on label_majority so the split stays the same whichever conflict treatment is used
# fold = -1 means test set, 0 to 4 are the development folds

from sklearn.model_selection import train_test_split, StratifiedKFold

dev_idx, test_idx = train_test_split(np.arange(len(vec)), test_size=0.2,
                                     stratify=vec['label_majority'], random_state=SEED)

vec['fold'] = -1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
y_dev = vec.loc[dev_idx, 'label_majority']
for fold_num, (_, val_pos) in enumerate(skf.split(dev_idx, y_dev)):
    vec.loc[dev_idx[val_pos], 'fold'] = fold_num

print("Development:", (vec['fold'] >= 0).sum(), " Test:", (vec['fold'] == -1).sum())
print(pd.crosstab(vec['label_majority'], vec['fold']))

Development: 122947  Test: 30737
fold               -1      0      1      2      3      4
label_majority                                          
0               17119  13695  13695  13695  13695  13695
1                1693   1355   1354   1354   1354   1355
2                  71     57     56     56     56     57
3                 857    686    686    685    686    686
4                5344   4275   4275   4275   4275   4274
5                  90     72     72     72     73     72
6                3782   3025   3026   3025   3025   3025
7                  32     26     27     26     26     26
8                 290    232    232    233    232    232
9                1459   1167   1167   1168   1167   1167


In [6]:
# save the benchmark table - every later notebook loads this one file
# service '?' and proto are left as they are, those get handled inside the fold loop like in A2

OUT_PATH = DATA_DIR + 'networkTraffic_vectors.csv'
vec.to_csv(OUT_PATH, index=False)

# reload to make sure nothing changed on the way to disk
check = pd.read_csv(OUT_PATH)
print("Saved shape:", vec.shape, " Reloaded shape:", check.shape)
print("Fold counts match:", (check['fold'].value_counts().sort_index() == vec['fold'].value_counts().sort_index()).all())
print("Labels match:", (check['label_majority'] == vec['label_majority']).all())
print("Missing values after reload:", check.isna().sum().sum())
print("\nColumns:", list(check.columns))

Saved shape: (153684, 47)  Reloaded shape: (153684, 47)
Fold counts match: True
Labels match: True
Missing values after reload: 0

Columns: ['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'label_first', 'label_majority', 'is_conflict', 'n_rows', 'fold']
